In [1]:
import pandas as pd
import numpy as np
import scipy.stats as sp
import statsmodels.api as sm

In [2]:
df = pd.read_csv("../data/data_1.csv")

If we did not have the RCT, our observations may contain an omitted variable bias. The impact of teacher gender on student performance may be correlated with other teacher characteristics. Let's assume that on average, female teachers have more experience at teaching than others. The estimated causal effect would partially capture experience - not just gender, biasing our β1 upwards.

Non-random sorting of students will also bias the observed results. If weaker students are disproportionately assigned to female teachers, then the estimated effect of gender will be correlated with the unobserved student ability. This biases our coefficient down, as weaker students perform worse, understating the true causal effect.

Reverse causality may also arise if schools assign female teachers to classrooms where students are underperforming. This would create a correlation between teacher gender and expected performance, as the classrooms ability will influence the teacher assignment rather than the other way around.

In [3]:
df.columns

Index(['stdntid', 'blockid', 'classid', 'tfa', 'classsize', 'femstdnt',
       'blkstdnt', 'hstdnt', 'wstdnt', 'pre_ts', 'femteach', 'blkteach',
       'hteach', 'wteach', 'teach_exp', 'teach_cert', 'ts'],
      dtype='object')

In [4]:
male = df[df["femteach"]==0]
female = df[df["femteach"]==1]

In [5]:
variables = ["femstdnt", "classsize", "blkstdnt", "hstdnt", "pre_ts"]

balance_test = []

for var in variables:
    mean_male = male[var].mean()
    mean_female = female[var].mean()
    
    t_stat, p_val = sp.ttest_ind(male[var], female[var])
    balance_test.append((var, mean_male, mean_female, p_val))
    
balance_df = pd.DataFrame(
    balance_test, 
    columns=["Variable", "Male Mean", "Female Mean", "p-value"])

balance_df["Significant (5%)"] = balance_df["p-value"] < 0.05
balance_df

,Variable,Male Mean,Female Mean,p-value,Significant (5%)
0,femstdnt,0.477833,0.489338,0.661743,False
1,classsize,23.702791,23.666667,0.841481,False
2,blkstdnt,0.238095,0.240180,0.926035,False
3,hstdnt,0.226601,0.207632,0.380438,False
4,pre_ts,50.385712,49.808580,0.280899,False


In [6]:
df["interaction"] = df["femstdnt"] * df["femteach"]

y = df["ts"]
x = df[["femstdnt", "femteach", "interaction", "blkstdnt", "hstdnt",
    "pre_ts", "classsize", "tfa"]]

x = sm.add_constant(x)
model = sm.OLS(y, x).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     ts   R-squared:                       0.651
Model:                            OLS   Adj. R-squared:                  0.650
Method:                 Least Squares   F-statistic:                     348.3
Date:                Sun, 01 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:50:54   Log-Likelihood:                -4487.8
No. Observations:                1500   AIC:                             8994.
Df Residuals:                    1491   BIC:                             9041.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           2.6698      1.117      2.391      0.017       0.479       4.860
femstdnt        2.6069      0.393      6.626      0.000       1.835       3.379
femteach        1.6553      0.354      4.673      0.000       0.961       2.350
interaction     2.9269      0.510      5.741      0.000       1.927       3.927
blkstdnt       -0.9772      0.293     -3.332      0.001      -1.552      -0.402
hstdnt          0.5996      0.304      1.972      0.049       0.003       1.196
pre_ts          0.5884      0.012     47.855      0.000       0.564       0.613
classsize      -0.0181      0.036     -0.498      0.619      -0.090       0.053
tfa            -0.2236      0.251     -0.892      0.372      -0.715       0.268
==============================================================================
Omnibus:                        2.480   Durbin-Watson:                   1.997
Prob(Omnibus):                  0.289   Jarque-Bera (JB):                2.332
Skew:                           0.040   Prob(JB):                        0.312
Kurtosis:                       2.824   Cond. No.                         508.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Students previous test scores are an indicator of inate ability, including it will absorb the effects of ability differences. The variable pre_ts is used to reduce the variance of our error term, ensuring that we're estimating the effect on learning gains and not just levels.

β$_{3}$ captures the differential effect a female teacher has on a female student relative to male students.

Although the balance test suggests the variables are not statistically different across groups, including them remains beneficial to our estimate. Even though our test has randomised controls, these may variables strongly predict post-treatment test scores. Controlling for them helps reduce the residual varianceand protects against small imbalances that may not be detected in the balance test.

Since the randomisation was conducted at the block level, we can interpret γ as the causal effect of assignment to a TFA teacher relative to non-TFA teachers.

In [7]:
df["teach_exp_sqr"] = df["teach_exp"] * df["teach_exp"]

x = df[["femstdnt", "femteach", "interaction", 
        "blkstdnt", "hstdnt", "pre_ts", "teach_exp",
        "teach_exp_sqr", "teach_cert", "blkteach",
        "hteach", "classsize", "tfa"]]

x = sm.add_constant(x)
model = sm.OLS(y, x).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     ts   R-squared:                       0.662
Model:                            OLS   Adj. R-squared:                  0.659
Method:                 Least Squares   F-statistic:                     223.4
Date:                Sun, 01 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:50:54   Log-Likelihood:                -4465.8
No. Observations:                1500   AIC:                             8960.
Df Residuals:                    1486   BIC:                             9034.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.3232      1.189      0.272      0.786      -2.010       2.656
femstdnt          2.7144      0.389      6.980      0.000       1.952       3.477
femteach          1.7154      0.350      4.902      0.000       1.029       2.402
interaction       2.8093      0.504      5.575      0.000       1.821       3.798
blkstdnt         -0.9272      0.290     -3.195      0.001      -1.496      -0.358
hstdnt            0.6500      0.301      2.162      0.031       0.060       1.240
pre_ts            0.5871      0.012     48.346      0.000       0.563       0.611
teach_exp         0.4032      0.090      4.466      0.000       0.226       0.580
teach_exp_sqr    -0.0134      0.004     -3.204      0.001      -0.022      -0.005
teach_cert        0.2714      0.276      0.984      0.325      -0.269       0.812
blkteach         -0.1044      0.309     -0.338      0.735      -0.710       0.501
hteach            0.3334      0.345      0.968      0.333      -0.342       1.009
classsize        -0.0272      0.036     -0.754      0.451      -0.098       0.043
tfa              -0.2016      0.247     -0.815      0.415      -0.687       0.284
==============================================================================
Omnibus:                        2.389   Durbin-Watson:                   2.002
Prob(Omnibus):                  0.303   Jarque-Bera (JB):                2.253
Skew:                           0.038   Prob(JB):                        0.324
Kurtosis:                       2.826   Cond. No.                     1.90e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.9e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

The regression table shows that our interaction term stayed relatively stable with the addition of teacher characteristics to our model, β$_{3}$ fell by just 0.1176, they do not explain the gender matching effect. The interaction term was not strongly correlated with omitted teacher characteristics in our previous regression, so our model appears robust.

We additionally controls for teacher experience and its squared term. Experience is likely to influence teaching skills and therfore student outcomes, so we control for this to reduce residual variance. Its also important to understand that the effect experience has on teaching is not linear, early years may yield greater improvements in effectiveness than later years. To capture this, we include its square term to allow for diminishing returns.

In [13]:
block_dummies = pd.get_dummies(df["blockid"], drop_first=True)

x_fe = df[["femstdnt", "femteach", "interaction", 
        "blkstdnt", "hstdnt", "pre_ts", "teach_exp",
        "teach_exp_sqr", "teach_cert", "blkteach",
        "hteach", "classsize", "tfa"]]

x_fe = pd.concat([x_fe, block_dummies], axis=1)

x_fe = sm.add_constant(x_fe)
fe = sm.OLS(y, x_fe).fit()
fe.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     ts   R-squared:                       0.675
Model:                            OLS   Adj. R-squared:                  0.661
Method:                 Least Squares   F-statistic:                     48.08
Date:                Mon, 02 Mar 2026   Prob (F-statistic):          8.88e-303
Time:                        11:19:58   Log-Likelihood:                -4435.8
No. Observations:                1500   AIC:                             8998.
Df Residuals:                    1437   BIC:                             9332.
Df Model:                          62                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.5743      1.466      0.392      0.695      -2.301       3.449
femstdnt          2.7689      0.396      6.999      0.000       1.993       3.545
femteach          1.6176      0.355      4.558      0.000       0.921       2.314
interaction       2.9548      0.512      5.776      0.000       1.951       3.958
blkstdnt         -1.0780      0.294     -3.666      0.000      -1.655      -0.501
hstdnt            0.6245      0.304      2.053      0.040       0.028       1.221
pre_ts            0.5895      0.012     48.187      0.000       0.566       0.614
teach_exp         0.3791      0.091      4.144      0.000       0.200       0.558
teach_exp_sqr    -0.0128      0.004     -3.025      0.003      -0.021      -0.004
teach_cert        0.3885      0.279      1.394      0.164      -0.158       0.935
blkteach         -0.1647      0.313     -0.526      0.599      -0.779       0.449
hteach            0.2459      0.348      0.706      0.480      -0.437       0.929
classsize        -0.0330      0.037     -0.904      0.366      -0.105       0.039
tfa              -0.1979      0.251     -0.789      0.430      -0.690       0.294
2                -0.9911      1.136     -0.872      0.383      -3.220       1.238
3                -0.8652      1.191     -0.726      0.468      -3.202       1.471
4                -1.5237      1.230     -1.239      0.216      -3.937       0.889
5                -1.0822      1.144     -0.946      0.344      -3.325       1.161
6                 0.7337      1.192      0.616      0.538      -1.604       3.072
7                 0.1876      1.274      0.147      0.883      -2.311       2.686
8                 0.4691      1.153      0.407      0.684      -1.792       2.730
9                -0.1162      1.181     -0.098      0.922      -2.432       2.200
10                1.5975      1.218      1.312      0.190      -0.791       3.986
11               -0.5134      1.182     -0.434      0.664      -2.831       1.804
12               -0.5999      1.193     -0.503      0.615      -2.940       1.740
13               -0.3945      1.293     -0.305      0.760      -2.932       2.143
14               -2.4218      1.192     -2.031      0.042      -4.760      -0.083
15                0.5626      1.274      0.442      0.659      -1.937       3.062
16                1.2883      1.277      1.009      0.313      -1.216       3.793
17                0.9131      1.149      0.795      0.427      -1.341       3.167
18               -1.4216      1.278     -1.112      0.266      -3.928       1.085
19               -0.1640      1.219     -0.135      0.893      -2.555       2.227
20                0.6948      1.246      0.558      0.577      -1.749       3.138
21               -0.3110      1.196     -0.260      0.795      -2.657       2.035
22                1.2594      1.169      1.077   

η$_{b}$ controls for between-block variation, it absorbs any unobserved factors common to students in the same block, such as school quality. The estimated coefficients are then identified using within-block variation.

Because β$_{3}$ remains stable after controlling for block fixed effects, the gender matching effect is identified from within-block variation and is unlikely to be driven by unobserved block-level confounding. Infact, β$_{3}$ increases slightly, insinuating that between-block differences slightly reduced the effect. Overall, the results are robust to the inclusion of block fixed effects.